# RFX2 pilot (Colab T4): xView RF-DETR fine-tuned at 2x nearest upscale
Declared in experiments/LOG.md. Runtime > Change runtime type > **T4 GPU**, then Run all. Checkpoints + predictions are saved to Google Drive (`MyDrive/auric_rfx2/`) so a disconnect loses at most one epoch.

In [ ]:
from google.colab import drive; drive.mount('/content/drive')
import os; OUT='/content/drive/MyDrive/auric_rfx2'; os.makedirs(OUT, exist_ok=True)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!pip -q install 'rfdetr[train]==1.11.2'
!python -c 'import torch_linear_assignment; print("tla ok")' || echo 'tla missing: script uses SciPy fallback'

In [ ]:
%%bash
set -e
cd /content
# Upload rfx_colab_data.zip (from the Mac: ~/auric_kaggle/colab/pack/) to MyDrive root first.
cp /content/drive/MyDrive/rfx_colab_data.zip /content/ && unzip -q -o rfx_colab_data.zip -d /content/data && rm rfx_colab_data.zip
wget -q -O ckpt1008.pth https://huggingface.co/ShotaroFukunaga/rfdetr_xview/resolve/main/1008/checkpoint0099.pth
sha256sum ckpt1008.pth  # expect cd5f2d2c...
ls /content/data; ls /content/data/tiles/images | wc -l; ls /content/data/val/images | wc -l

In [ ]:
import glob, os
CODE, TILES = '/content/data/code', '/content/data/tiles'
os.makedirs('/content/valroot', exist_ok=True)
if not os.path.exists('/content/valroot/val'): os.symlink('/content/data/val', '/content/valroot/val')
print(len(glob.glob('/content/valroot/val/images/*.png')), 'val images (expect 22)')
os.environ.update(RFX_CKPT='/content/ckpt1008.pth', CODE=CODE, TILES=TILES, OUT=OUT)

In [ ]:
# Train: 2x upscale, 3 epochs max, 2.5 h cap, checkpoint every epoch (to Drive)
!cd $CODE && python rfx_finetune.py --tiles $TILES --out $OUT --upscale 2 --epochs 3 --bs 4 --accum 2 --save_from 1 --hours 2.5 2>&1 | grep -v -i warn

In [ ]:
# Predict val at 2x with the last checkpoint -> CSV in Drive (scored on the Mac with evaluate.py)
import glob
ck = sorted(glob.glob(OUT + '/rfx1_ep*.pt'), key=lambda p: int(p.split('ep')[-1][:-3]))[-1]; print(ck)
!cd $CODE && python rfx_predict.py --weights {ck} --root /content/valroot --scale 2 --out $OUT/RFX2_val_preds.csv 2>&1 | tail -3
!ls -la $OUT